### «Умный помощник» для оформления командировок

#### Установка зависимостей

In [1]:
%pip install -q langchain langgraph langchain-community chromadb sentence-transformers torch pandas openai python-dotenv langchain-openai

Note: you may need to restart the kernel to use updated packages.


#### Конфигурация

In [2]:
import os
from typing import Dict, List, Any
import pandas as pd
# from io import StringIO
import traceback
from dotenv import load_dotenv

# Загружаем .env
load_dotenv()

# === Чтение конфигурации ===
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openrouter").lower()

# OpenRouter
OPENROUTER_API_KEY = os.getenv("API_KEY", "")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-20b:free")

# Локальная (Ollama / LM Studio)
# LOCAL_MODEL_NAME = os.getenv("LOCAL_MODEL_NAME", "llama3.2")
# LOCAL_BASE_URL = os.getenv("LOCAL_BASE_URL", "http://localhost:11434")
# LOCAL_API_KEY = os.getenv("LOCAL_API_KEY", "ollama")

# Эмбеддинги
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

# Логирование
LOG_LEVEL = os.getenv("LOG_LEVEL", "info").upper()

# === Инициализация LLM ===
llm = None

if LLM_PROVIDER == "openrouter" and OPENROUTER_API_KEY:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        api_key=OPENROUTER_API_KEY, # type: ignore
        base_url=OPENROUTER_BASE_URL,
        model=OPENROUTER_MODEL,
        temperature=0.7
    )
    print(f"✅ LLM инициализирован: OpenRouter / {OPENROUTER_MODEL}")
    # Быстрый пинг
    try:
        response = llm.invoke("Ответь OK")
        print(f"✅ Пинг успешен: {response.content[:50]}")
    except Exception as e:
        print(f"❌ Пинг провален: {e}")
        
        
else:
    print("⚠️ LLM не сконфигурирован (нет API ключа или выбран неподдерживаемый провайдер).")
    print("   Будет использована заглушка DummyLLM, которая не генерирует осмысленные ответы.")
    class DummyLLM:
        def invoke(self, prompt, **kwargs):
            return f"[DummyLLM] Нет реального LLM. Промпт: {prompt[:200]}..."
    llm = DummyLLM()

def print_config():
    print("\n" + "="*50)
    print("ТЕКУЩАЯ КОНФИГУРАЦИЯ")
    print("="*50)
    print(f"LLM Provider: {LLM_PROVIDER.upper()}")
    if LLM_PROVIDER == "openrouter":
        print(f"Model: {OPENROUTER_MODEL}")
        print(f"API Key: {'***' if OPENROUTER_API_KEY else 'НЕТ'}")
    # elif LLM_PROVIDER == "local":
    #     print(f"Model: {LOCAL_MODEL_NAME}")
    #     print(f"Base URL: {LOCAL_BASE_URL}")
    print(f"Embedding: {EMBEDDING_MODEL}")
    print("="*50 + "\n")

print_config()

c:\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM инициализирован: OpenRouter / openai/gpt-oss-20b:free
❌ Пинг провален: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'openai/gpt-oss-20b:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'OpenInference', 'is_byok': False}}, 'user_id': 'user_3EVG4RRUhE3TcCeo6e305BQMFXX'}

ТЕКУЩАЯ КОНФИГУРАЦИЯ
LLM Provider: OPENROUTER
Model: openai/gpt-oss-20b:free
API Key: ***
Embedding: sentence-transformers/all-MiniLM-L6-v2



#### из файлов

In [3]:
# import pandas as pd
import sqlite3
# from typing import List, Dict, Any

class FlightSearch:
    def __init__(self, csv_path: str = 'flights.csv'):
        # Загружаем CSV 
        self.df = pd.read_csv(csv_path, sep=',')
        print("Колонки в загруженном CSV:", list(self.df.columns))

        # Создаём in-memory SQLite
        self.conn = sqlite3.connect(':memory:')
        self.df.to_sql('flights', self.conn, index=False, if_exists='replace')
        # Для удобства сохраняем курсор
        self.cursor = self.conn.cursor()
    
    # def search_direct(self, origin: str, destination: str, date: str) -> pd.DataFrame:
    def search_direct(self, origin: str, destination: str, date: str) -> List[Dict[str, Any]]:
        """
        Поиск прямых рейсов по точному совпадению городов и даты.
        origin, destination — названия городов (как в файле)
        date — строка в формате 'YYYY-MM-DD'
        """
        query = """
            SELECT line, flight, departure, arrival, departure_date, price
            FROM flights
            WHERE departure = ? AND arrival = ? AND departure_date = ?
            ORDER BY price
        """

        flights = pd.read_sql_query(query, self.conn, params=(origin, destination, date))

        results = []

        for _, flight in flights.iterrows():
            results.append({
                'flight': {
                    'line': flight['line'],
                    'flight': flight['flight'],
                    'departure': flight['departure'],
                    'arrival': flight['arrival'],
                    'date': flight['departure_date'],                    
                },
                'price': flight['price']
            })

        return results
        # return pd.read_sql_query(query, self.conn, params=(origin, destination, date))
    
    def search_connecting(self, origin: str, destination: str, date: str, max_layover_days: int = 1) -> List[Dict[str, Any]]:
        """
        Поиск стыковочных рейсов (с одной пересадкой).
        Первый рейс вылетает точно в указанную дату.
        Второй рейс вылетает из города пересадки не ранее даты первого рейса
        и не позже date + max_layover_days.
        Возвращает список маршрутов с информацией о пересадке.
        """
        # Шаг 1: Найти все возможные города пересадки (куда можно улететь из origin в нужную дату)
        query_first = """
            SELECT DISTINCT arrival AS stopover
            FROM flights
            WHERE departure = ? AND departure_date = ?
        """
        stopovers = pd.read_sql_query(query_first, self.conn, params=(origin, date))['stopover'].tolist()
        
        if not stopovers:
            return []
        
        # Шаг 2: Для каждого города пересадки ищем рейсы до destination
        # Формируем условие для даты второго рейса: между date и date+max_layover_days
        date_end = pd.to_datetime(date) + pd.Timedelta(days=max_layover_days)
        date_end_str = date_end.strftime('%Y-%m-%d')
        
        results = []
        for stop in stopovers:
            query_second = """
                SELECT line, flight, departure, arrival, departure_date, price
                FROM flights
                WHERE departure = ? AND arrival = ? AND departure_date BETWEEN ? AND ?
                ORDER BY departure_date, price
            """
            second_flights = pd.read_sql_query(query_second, self.conn, params=(stop, destination, date, date_end_str))
            if second_flights.empty:
                continue
            
            # Получаем первый рейс (их может быть несколько)
            first_flights = pd.read_sql_query(
                "SELECT * FROM flights WHERE departure = ? AND arrival = ? AND departure_date = ?",
                self.conn, params=(origin, stop, date)
            )
            
            for _, first in first_flights.iterrows():
                for _, second in second_flights.iterrows():
                    # Проверяем, что второй рейс не раньше первого (по дате)
                    if second['departure_date'] < first['departure_date']:
                        continue
                    results.append({
                        'first_flight': {
                            'line': first['line'],
                            'flight': first['flight'],
                            'departure': first['departure'],
                            'arrival': first['arrival'],
                            'date': first['departure_date'],
                            'price': first['price']
                        },
                        'second_flight': {
                            'line': second['line'],
                            'flight': second['flight'],
                            'departure': second['departure'],
                            'arrival': second['arrival'],
                            'date': second['departure_date'],
                            'price': second['price']
                        },
                        'stopover_city': stop,
                        'price': first['price'] + second['price']
                    })
        
        # Сортируем по суммарной цене
        results.sort(key=lambda x: x['price'])
        return results
    
    def show_all_connecting(self) -> List[Dict[str, Any]]:
        """
        Все возможные стыковочные маршруты с одной пересадкой, где
        дата второго рейса не раньше даты первого и разница не более 2 дней.
        Возвращает список словарей с информацией о каждом стыковочном маршруте.
        """
        query = """
            SELECT 
                f1.line AS line1,
                f1.flight AS flight1,
                f1.departure AS from_city,
                f1.arrival AS stopover_city,
                f1.departure_date AS date1,
                f1.price AS price1,
                f2.line AS line2,
                f2.flight AS flight2,
                f2.departure AS stopover_city2,
                f2.arrival AS to_city,
                f2.departure_date AS date2,
                f2.price AS price2,
                (f1.price + f2.price) AS total_price
            FROM flights f1
            JOIN flights f2 ON f1.arrival = f2.departure
            WHERE 
                f2.departure_date >= f1.departure_date
                AND julianday(f2.departure_date) - julianday(f1.departure_date) <= 2
                AND f1.departure != f2.arrival   -- опционально: исключаем возврат в исходный город
            ORDER BY total_price
        """
        df = pd.read_sql_query(query, self.conn)
        
        results = []
        for _, row in df.iterrows():
            results.append({
                'first_flight': {
                    'line': row['line1'],
                    'flight': row['flight1'],
                    'departure': row['from_city'],
                    'arrival': row['stopover_city'],
                    'date': row['date1'],
                    'price': row['price1']
                },
                'second_flight': {
                    'line': row['line2'],
                    'flight': row['flight2'],
                    'departure': row['stopover_city2'],
                    'arrival': row['to_city'],
                    'date': row['date2'],
                    'price': row['price2']
                },
                'stopover_city': row['stopover_city'],
                'price': row['total_price']
            })
        return results

        
    
    def close(self):
        self.conn.close()

In [4]:
def print_direct_flights(direct):
    for route in direct:  
        print(f"{route['flight']['flight']} {route['flight']['departure']}→{route['flight']['arrival']} ({route['flight']['date']})"          
              f"Цена: {route['price']} руб.")

def print_connecting_flights(all_conn):
    for route in all_conn: 
        print(f"{route['first_flight']['flight']} {route['first_flight']['departure']}→{route['first_flight']['arrival']} ({route['first_flight']['date']}) + "
            f"{route['second_flight']['flight']} {route['second_flight']['departure']}→{route['second_flight']['arrival']} ({route['second_flight']['date']})  "
            f"Цена: {route['price']} руб.")
        

fs = FlightSearch('data/flights.csv')

# Прямые рейсы из Москвы в Санкт-Петербург 2026-01-14
direct = fs.search_direct('Москва', 'Санкт-Петербург', '2026-01-14')
print("✈ Прямые рейсы из Москвы в Санкт-Петербург 2026-01-14:")
print_direct_flights(direct)

# Стыковочные рейсы
all_conn = fs.show_all_connecting()
print(f"\n✈✈ Найдено стыковочных маршрутов: {len(all_conn)}")

print("Стыковочные маршруты:")
print_connecting_flights(all_conn)

# Загрузка политики из data/policy.txt
with open("data/policy.txt", "r", encoding="utf-8") as f:
    policy_text = f.read()
print(f"\n✅ Загружен текст политики ({len(policy_text)} символов) из data/policy.txt")


Колонки в загруженном CSV: ['line', 'flight', 'departure', 'arrival', 'departure_date', 'price']
✈ Прямые рейсы из Москвы в Санкт-Петербург 2026-01-14:
РО5925 Москва→Санкт-Петербург (2026-01-14)Цена: 6005 руб.

✈✈ Найдено стыковочных маршрутов: 97
Стыковочные маршруты:
РО5067 Москва→Иркутск (2026-12-08) + УР7353 Иркутск→Волгоград (2026-12-09)  Цена: 7854 руб.
ПО3046 Волгоград→Москва (2026-09-18) + ПО3624 Москва→Санкт-Петербург (2026-09-19)  Цена: 9489 руб.
НО6518 Волгоград→Иркутск (2026-03-15) + ПО3138 Иркутск→Воркута (2026-03-16)  Цена: 10054 руб.
ПО3046 Волгоград→Москва (2026-09-18) + ЮТ4540 Москва→Воркута (2026-09-18)  Цена: 10842 руб.
АЭ9999 Санкт-Петербург→Москва (2026-07-15) + РО8888 Москва→Воркута (2026-07-16)  Цена: 11700 руб.
УР7263 Иркутск→Волгоград (2026-04-17) + АЭ1641 Волгоград→Москва (2026-04-17)  Цена: 11814 руб.
НО6906 Иркутск→Санкт-Петербург (2026-06-16) + РО5774 Санкт-Петербург→Воркута (2026-06-18)  Цена: 12318 руб.
АЭ1283 Волгоград→Москва (2026-05-24) + ПО3423 Москва

#### Embeddings

In [5]:
import shutil
import os

if os.path.exists('.data/.chroma'):
    shutil.rmtree('.data/.chroma')
    print("предыдущая бд удалена")

предыдущая бд удалена


In [6]:
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma


# Эмбеддер
try:
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )
    print("✅ Эмбеддер загружен")
except Exception as e:
    print(f"⚠️ Ошибка загрузки эмбеддера: {e}")
    class DummyEmbeddings:
        def embed_documents(self, texts): return [[0.0]*384 for _ in texts]
        def embed_query(self, text): return [0.0]*384
    embeddings = DummyEmbeddings()
    print("⚠️ Используется DummyEmbeddings")

# Документы
policy_doc = Document(page_content=policy_text, metadata={"source": "policy"})

# ticket_docs = []
# for _, row in flights_itab.iterrows():
#     content = (
#         f"Рейс {row['flight']} "
#         f"{row['line']} "
#         f"{row['departure']} → {row['arrival']} "
#         f"{row['departure_date']}, "
#         f"{row['price']} руб., "
#         # f"{'прямой' if row['is_direct'] else 'с пересадкой'}"
#     )
#     metadata = {
#         "source": "flight",
#         "flight_number": row["flight"],
#         "departure_city": row["departure"],
#         "arrival_city": row["arrival"],
#         "price": row["price"],
#         "departure_date": row["departure_date"]
#     }
#     ticket_docs.append(Document(page_content=content, metadata=metadata))

# Чанкинг только для политики
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
policy_chunks = splitter.split_documents([policy_doc])
print(f"Политика разбита на {len(policy_chunks)} чанков")

# Векторное хранилище policy
# vectorstore = Chroma.from_documents(documents= policy_chunks + ticket_docs,
vectorstore = Chroma.from_documents(documents= policy_chunks,
                                    embedding=embeddings, # type: ignore
                                    persist_directory='.data/.chroma' )
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ policy vector store готов")

z:\temp\ipykernel_7592\792820939.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
z:\temp\ipykernel_7592\792820939.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13418.65it/s]


✅ Эмбеддер загружен
Политика разбита на 17 чанков
✅ policy vector store готов


#### LangGraph

In [7]:
# import re
from datetime import datetime, timedelta
from typing import Optional, List, Dict
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
from typing import TypedDict

class ParsedTravelQuery(BaseModel):
    departure: Optional[str] = Field(None, description="Город вылета")
    arrival: Optional[str] = Field(None, description="Город прилёта")
    departure_date: Optional[str] = Field(
        None, description="Дата вылета в формате YYYY-MM-DD"
    )    

class AgentState(TypedDict):
    user_query: str
    parsed_query: Dict    
    policy_context: List[str]
    flights: List[Dict]
    budget_ok: bool
    final_answer: str

class QueryParser:
    """Извлекает города и дату из запроса LLM"""

    def __init__(self, llm):
        self.llm = llm

    def invoke(self, state: AgentState) -> AgentState:
        print("🍌Агент QueryParser: извлечение параметров ")
        query = state["user_query"]

        # Вызов LLM
        parsed = self._llm_extract(query)
        
        state["parsed_query"] = parsed
        print(f"🍤Распознано: {parsed}")
        return state

    def _llm_extract(self, query: str) -> Dict:
        """Вызов LLM с Pydantic-парсером."""
        # from langchain_core.output_parsers import PydanticOutputParser

        parser = PydanticOutputParser(pydantic_object=ParsedTravelQuery)
        prompt = f"""
Ты – помощник по поиску билетов на авиасейлс. Извлеки из запроса: город вылета, город прилёта, дату вылета.

Запрос: {query}

{parser.get_format_instructions()}

Правила:
- Города пиши в именительном падеже (Москва, Санкт-Петербург).
- Дату приведи в формат YYYY-MM-DD. Если дата относительная ("завтра"), считай от {datetime.now().date()}.
- Ничего не выдумывай
- Если город не указан, считай, что это Москва.
- Если дата не указана, считай, что это {datetime.now().date() + timedelta(days=1)}.
"""
        try:
            resp = self.llm.invoke(prompt)
            parsed_obj = parser.parse(resp.content)
            return parsed_obj.model_dump()
        except Exception as e:
            print(f"   Ошибка LLM: {e}")
            return {}
        
# ---------- Агент 001: Поиск билетов ----------
class TicketSearcher:
    """ Поиск подходящих рейсов по городам и дате в SQLite
    """
    def invoke(self, state: AgentState) -> AgentState:
        print("🔍 Агент TicketSearcher: поиск билетов двум городам и дате")
        parsed = state.get("parsed_query", {})
        from_city = parsed.get("departure")
        to_city = parsed.get("arrival")
        date_str = parsed.get("departure_date")

        if not from_city or not to_city:
            print("💫 Города не определены, поиск отменён.")
            state["flights"] = []
            return state

        if not date_str:
            print("💫 Дата не определена, поиск отменён.")
            state["flights"] = []
            return state        

        routes = fs.search_direct(from_city, to_city, date_str)
        if len(routes) > 0:
            print("Прямые рейсы:")
            print_direct_flights(routes)
        else: 
            print("Прямых рейсов не найдено.")

        if len(routes) == 0:
            routes = fs.search_connecting(from_city, to_city, date_str)
            print(f"✅ Найдено стыковочных маршрутов: {len(routes)}")
            print_connecting_flights(routes)

        state["flights"] = routes
        print(f"   Найдено билетов: {len(state['flights'])}")
        return state
    
# ---------- Агент 002: Поиск политики ----------
class PolicyExpert:
    def invoke(self, state: AgentState) -> AgentState:
        print("📋 Агент PolicyExpert: поиск релевантных правил...")
        
        # Ищем ТОЛЬКО документы с source=policy
        policy_docs = vectorstore.similarity_search(
            state["user_query"], 
            k=3, 
            filter={"source": "policy"}
        )
        
        state["policy_context"] = [d.page_content for d in policy_docs]
        print(f"   Найдено {len(state['policy_context'])} фрагментов политики")
        
        # Fallback: если ничего не нашлось, берём общую выдачу и фильтруем
        if not state["policy_context"]:
            all_docs = retriever.invoke(state["user_query"])
            state["policy_context"] = [d.page_content for d in all_docs if d.metadata.get("source") == "policy"]
            
        return state    

# ---------- Агент 003: Бюджетный контроль ----------
class BudgetAnalyst:
    def invoke(self, state: AgentState) -> AgentState:
        print("💰 Агент BudgetAnalyst: проверка лимитов...")
        if not state["flights"]:
            state["budget_ok"] = False
            return state
        # Простейший лимит из политики (можно вытащить из policy_context)
        MAX_PRICE = 50000
        state["budget_ok"] = any(t["price"] <= MAX_PRICE for t in state["flights"])
        print(f"   Бюджет {'соблюдён' if state['budget_ok'] else 'превышен'}")
        return state
    
# ---------- Агент 004: Генерация ответа через LLM ----------
class FinalAnswerGenerator:
    def __init__(self, llm):
        self.llm = llm

    def invoke(self, state: AgentState) -> AgentState:
        print("🤖 Агент FinalAnswerGenerator: формирование ответа с помощью LLM...")
        # Собираем контекст для LLM
        context = {
            "query": state["user_query"],
            "policy": (
                "\n".join(state["policy_context"])
                if state["policy_context"]
                else "(нет данных)"
            ),
            "flights": state["flights"],
            "budget_ok": state["budget_ok"],
        }

        prompt = f"""Ты — помощник по по поиску авиарейсов. Ответь пользователю на русском языке, профессионально и лаконично.

Запрос пользователя: {context['query']}

Информация из политики компании:
{context['policy']}

Найденные билеты:
{context['flights']}
Соответствие бюджету: {'Да' if context['budget_ok'] else 'Нет (превышает лимит 50 000 руб)'}
Сформируй понятный, дружелюбный ответ, который суммирует информацию. Если билетов нет — сообщи об этом. Ничего не выдумывай."""

        try:
            response = self.llm.invoke(prompt)
            state["final_answer"] = (
                response.content if hasattr(response, "content") else str(response)
            )
        except Exception as e:
            print(f"   Ошибка LLM: {e}")
            state["final_answer"] = (
                f"Извините, произошла ошибка при генерации ответа. Детали: {e}"
            )

        return state    

In [8]:
workflow = StateGraph(AgentState)

workflow.add_node("query_parser", QueryParser(llm).invoke)
workflow.add_node("ticket_searcher", TicketSearcher().invoke)
workflow.add_node("policy_expert", PolicyExpert().invoke)
workflow.add_node("budget_analyst", BudgetAnalyst().invoke)
workflow.add_node("final_answer", FinalAnswerGenerator(llm).invoke)


workflow.set_entry_point("query_parser")
workflow.add_edge("query_parser", "ticket_searcher")
workflow.add_edge("ticket_searcher", "policy_expert")
workflow.add_edge("policy_expert", "budget_analyst")
workflow.add_edge("budget_analyst", "final_answer")
workflow.add_edge("final_answer", END)

app = workflow.compile()
print("✅ Граф агентов готов")

✅ Граф агентов готов


#### Тесты

In [9]:
def run_query(query: str):
    print(f"\n{'='*60}\nЗАПРОС: {query}\n{'='*60}")

# class AgentState(TypedDict):
#     user_query: str
#     parsed_query: Dict    
#     policy_context: List[str]
#     flights: List[Dict]
#     budget_ok: bool
#     final_answer: str

    initial_state = {
        "user_query": query,
        "parsed_query": {},      
        "policy_context": [],
        "flights": [],
        "budget_ok": False,
        "final_answer": ""
    }
    try:
        result = app.invoke(initial_state)
        print("\n✅ ОТВЕТ АССИСТЕНТА:\n")
        print(result["final_answer"])
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        traceback.print_exc()

In [10]:
# Тест 1: Поиск билетов
run_query("Нужны билеты из Москвы в Иркутск с вылетом 13 февраля 2026")

# Тест 2: Поиск стыковочного рейса
run_query("Нужны билеты из Санкт-Петербурга в Воркуту с вылетом 15 июля 2026")

# Тест 3: Вопрос о правилах
run_query("Можно ли есть экзотические блюда местной кухни в командировке?")

# Тест 4: Билет в город
run_query("Нужен билет в Воркуту")


ЗАПРОС: Нужны билеты из Москвы в Иркутск с вылетом 13 февраля 2026
🍌Агент QueryParser: извлечение параметров 
🍤Распознано: {'departure': 'Москва', 'arrival': 'Иркутск', 'departure_date': '2026-02-13'}
🔍 Агент TicketSearcher: поиск билетов двум городам и дате
Прямые рейсы:
АЭ1677 Москва→Иркутск (2026-02-13)Цена: 16539 руб.
   Найдено билетов: 1
📋 Агент PolicyExpert: поиск релевантных правил...
   Найдено 3 фрагментов политики
💰 Агент BudgetAnalyst: проверка лимитов...
   Бюджет соблюдён
🤖 Агент FinalAnswerGenerator: формирование ответа с помощью LLM...

✅ ОТВЕТ АССИСТЕНТА:

Здравствуйте!  
Для даты **13 февраля 2026 г.** у нас есть один вариант:

| Перелёт | Авиакомпания | Номер рейса | Откуда | Куда | Цена |
|---------|--------------|-------------|--------|------|------|
| Москва → Иркутск | Аэрофлот | АЭ1677 | Москва | Иркутск | 16 539 ₽ |

Стоимость соответствует вашему бюджету.  

Если вам нужна дополнительная информация (время вылета, наличие бизнес‑класса и т.д.) — дайте знать!

